# Cell 1: Imports and Configurations

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import math
import torch
import spacy
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- GPU SPEED OPTIMIZATIONS ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

BASE_DIR = "."
TRAIN_CSV = "train_dataset.csv"
TEST_CSV = "test_dataset.csv"

# Local output paths
LOCAL_TRAIN_VECS = "train_vectors_unified.pt"
LOCAL_TEST_VECS = "test_vectors_unified.pt"
LOCAL_TRAIN_TASKS = "train_tasks_expanded.csv"
LOCAL_TEST_TASKS = "test_tasks_expanded.csv"

# --- LOAD SPACY ---
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Cell 2: LLM Load

In [ ]:
# --- LOAD LLAMA ---
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
print(f"Loading {model_id} onto A100...")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16, attn_implementation="sdpa"
)
model.eval()
terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

# Cell 3: Helper Functions and Execution

In [ ]:
# --- EXTRACTION HELPERS ---
def get_log2_chunks(seq_len):
    if seq_len <= 1: return [(0, seq_len)]
    num_chunks = max(1, int(math.log2(seq_len)))
    chunk_size = seq_len / num_chunks
    chunks = []
    for i in range(num_chunks):
        start = int(i * chunk_size)
        end = int((i + 1) * chunk_size) if i < (num_chunks - 1) else seq_len
        chunks.append((start, end))
    return chunks

def get_entity_weights(text, token_offsets, seq_len):
    doc = nlp(text)
    weights = torch.full((seq_len,), 0.1, device=model.device, dtype=torch.bfloat16)

    entity_spans = [(ent.start_char, ent.end_char) for ent in doc.ents]
    noun_spans = [(chunk.start_char, chunk.end_char) for chunk in doc.noun_chunks]
    all_spans = entity_spans + noun_spans

    for idx, (tok_start, tok_end) in enumerate(token_offsets):
        if tok_start == tok_end: continue
        for (span_start, span_end) in all_spans:
            if tok_start < span_end and tok_end > span_start:
                weights[idx] = 1.0
                break
    return weights

# --- 6. CORE BATCHED PIPELINE ---
def process_dataset(input_csv, local_vecs_path, local_tasks_path, batch_size=16):
    print(f"\nProcessing {os.path.basename(input_csv)}...")
    df = pd.read_csv(input_csv)

    # Clean data using the new column names
    original_len = len(df)
    df.dropna(subset=['question_id', 'document_topic', 'context', 'question'], inplace=True)

    # ⚡ SPEED TRICK 1: Sort by length to drastically reduce padding waste
    df['approx_len'] = df['context'].str.len()
    df.sort_values('approx_len', inplace=True)
    df.drop(columns=['approx_len'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"Dropped {original_len - len(df)} rows. Clean rows remaining: {len(df)}")

    vectors_dict = {}
    expanded_rows = []
    target_layers = [8, 16, 24]

    for start_idx in tqdm(range(0, len(df), batch_size), desc="Processing Batches"):
        batch_df = df.iloc[start_idx:start_idx+batch_size]

        # ==========================================
        # PHASE A: TURBO TASK GENERATION
        # ==========================================
        # We define the 3 tasks with specific token limits so the GPU never waits
        task_definitions = [
            {
                "name": "QA",
                "sys": "Answer the question using only the provided context. Output a single, brief paragraph. No conversational filler.",
                "user_template": "Question: {q}\n Context: {c}",
                "inj_template": "Question: {q}\n Context: ...",
                "max_tokens": 100  # ⚡ SPEED TRICK 2: Hard cutoff for short tasks
            },
            {
                "name": "Summary",
                "sys": "Summarize the provided context in one short paragraph. Absolutely no bullet points or lists.",
                "user_template": "Tell me about: {c}",
                "inj_template": "Tell me about: ...",
                "max_tokens": 150
            },
            {
                "name": "Repeat",
                "sys": "Output the exact text provided without any additional commentary.",
                "user_template": "Repeat the context word by word like a paragraph: {c}",
                "inj_template": "Repeat the context word by word like a paragraph: ...",
                "max_tokens": 400
            }
        ]

        for task in task_definitions:
            gold_prompts = []
            row_metadata = []

            # 1. Build prompts for this specific task
            for _, row in batch_df.iterrows():
                gold_user = task["user_template"].format(q=row['question'], c=row['context'])
                inj_user = task["inj_template"].format(q=row['question'])

                gold_msg = [{"role": "system", "content": task["sys"]}, {"role": "user", "content": gold_user}]
                inj_msg = [{"role": "system", "content": task["sys"]}, {"role": "user", "content": inj_user}]

                gold_prompts.append(tokenizer.apply_chat_template(gold_msg, tokenize=False, add_generation_prompt=True))

                row_metadata.append({
                    "question_id": row['question_id'],
                    "document_topic": row['document_topic'],
                    "context": row['context'],
                    "task_name": task["name"],
                    "inj_prompt": tokenizer.apply_chat_template(inj_msg, tokenize=False, add_generation_prompt=True)
                })

            # 2. Generate for this specific task
            gold_inputs = tokenizer(gold_prompts, padding=True, return_tensors="pt").to(model.device)

            with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                gens = model.generate(
                    **gold_inputs,
                    max_new_tokens=task["max_tokens"],
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=terminators
                )

            # 3. Decode and save
            input_len = gold_inputs.input_ids.shape[1]
            for i, info in enumerate(row_metadata):
                info["target_response"] = tokenizer.decode(gens[i][input_len:], skip_special_tokens=True).strip()
                expanded_rows.append(info)

            # Clear cache between tasks to keep VRAM clean
            del gold_inputs, gens
            torch.cuda.empty_cache()

        # ==========================================
        # PHASE B: BATCHED MULTI-LAYER EXTRACTION
        # ==========================================
        contexts = batch_df['context'].astype(str).tolist()

        # 1. SWITCH PADDING TO RIGHT: Makes tensor slicing mathematically safe and fast
        tokenizer.padding_side = "right"
        ext_inputs = tokenizer(contexts, return_tensors="pt", padding=True, return_offsets_mapping=True).to(model.device)

        # 2. BATCHED FORWARD PASS: Do all 16 rows at once on the GPU!
        with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            ext_outputs = model(
                input_ids=ext_inputs.input_ids,
                attention_mask=ext_inputs.attention_mask,
                output_hidden_states=True
            )

        # 3. UNPACK THE BATCH
        for i, row in enumerate(batch_df.itertuples()):
            c_id = row.question_id
            seq_len = ext_inputs.attention_mask[i].sum().item()

            # Because we right-padded, valid tokens are perfectly aligned from 0 to seq_len
            offsets = ext_inputs.offset_mapping[i][:seq_len].tolist()
            weights = get_entity_weights(row.context, offsets, seq_len)

            chunk_boundaries = get_log2_chunks(seq_len)
            num_chunks = len(chunk_boundaries)

            chunk_tensor = torch.zeros((num_chunks, len(target_layers), 3, 4096), dtype=torch.bfloat16, device='cpu')
            global_last_tensor = torch.zeros((len(target_layers), 4096), dtype=torch.bfloat16, device='cpu')

            for l_idx, layer_num in enumerate(target_layers):
                # Grab the exact unpadded sequence from the batched tensor
                actual_hs = ext_outputs.hidden_states[layer_num][i, :seq_len, :]
                global_last_tensor[l_idx, :] = actual_hs[-1, :].cpu()

                for c_idx, (start, end) in enumerate(chunk_boundaries):
                    chunk_weights = weights[start:end].unsqueeze(-1)
                    w_sum = chunk_weights.sum().clamp(min=1e-6)

                    hs_chunk = actual_hs[start:end, :]

                    mu = (hs_chunk * chunk_weights).sum(dim=0) / w_sum
                    var = (chunk_weights * (hs_chunk - mu)**2).sum(dim=0) / w_sum
                    max_vec = (hs_chunk * chunk_weights).max(dim=0).values

                    chunk_tensor[c_idx, l_idx, 0, :] = mu.cpu()
                    chunk_tensor[c_idx, l_idx, 1, :] = var.cpu()
                    chunk_tensor[c_idx, l_idx, 2, :] = max_vec.cpu()

            vectors_dict[c_id] = {
                "chunk_signals": chunk_tensor,
                "global_last": global_last_tensor
            }

        # 4. SWITCH PADDING BACK TO LEFT FOR THE NEXT GENERATION BATCH
        tokenizer.padding_side = "left"

        # Free memory aggressively
        del ext_inputs, ext_outputs
        torch.cuda.empty_cache()

    # Save to high-speed local disk
    print(f"Writing outputs to local SSD...")
    torch.save(vectors_dict, local_vecs_path)
    pd.DataFrame(expanded_rows).to_csv(local_tasks_path, index=False)
    print(f"✅ Extracted {len(expanded_rows)} tasks and {len(vectors_dict)} vectors locally.")


In [ ]:
# --- 7. EXECUTE ---
process_dataset(TEST_CSV, LOCAL_TEST_VECS, LOCAL_TEST_TASKS, batch_size=128)

In [ ]:
process_dataset(TRAIN_CSV, LOCAL_TRAIN_VECS, LOCAL_TRAIN_TASKS, batch_size=128)